[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_Chronos/VaR_CEE_Chronos.ipynb)

# VaR_CEE_Chronos

Generates zero-shot VaR and ES forecasts using the Chronos-2 Bolt foundation model (Amazon).
Obtains 9 quantile forecasts (10th–90th percentiles) per date, fits a Student-t distribution
via quantile matching, and extracts VaR/ES analytically from the fitted distribution.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
# Install dependencies
!pip install -q chronos-forecasting torch pandas numpy matplotlib scipy

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import t as t_dist
from google.colab import files

np.random.seed(42)
torch.manual_seed(42)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}

OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
FM_CONTEXT = 250
FM_HORIZON = 1
STRIDE = 5

Q_LEVELS = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

# Output directory
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

print("Series to process:")
for c, raw_id, sid, st in get_all_series():
    print(f"  {c}: {sid} ({st})")

## 1. Upload Raw Data

Upload the 10 CSV files from `data/raw/`:
- `BET.csv`, `WIG20.csv`, `PX.csv`, `BUX.csv`, `SOFIX.csv`
- `EURRON.csv`, `EURPLN.csv`, `EURCZK.csv`, `EURHUF.csv`, `USDBGN.csv`

Each CSV must have columns: `Date, Open, High, Low, Close, Volume`

In [ ]:
# Upload CSV files
print("Please upload the 10 raw CSV files (BET.csv, WIG20.csv, PX.csv, BUX.csv, SOFIX.csv,")
print("EURRON.csv, EURPLN.csv, EURCZK.csv, EURHUF.csv, USDBGN.csv):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files: {list(uploaded.keys())}")

In [ ]:
# ── Compute log returns from raw CSVs ─────────────────────
returns_dict = {}

for country, raw_id, series_id, stype in get_all_series():
    csv_file = f"{raw_id}.csv"
    if csv_file not in uploaded:
        print(f"  WARNING: {csv_file} not uploaded, skipping {series_id}")
        continue

    df = pd.read_csv(csv_file, parse_dates=["Date"], index_col="Date")
    df = df.sort_index()
    # Log returns from Close prices
    log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
    log_ret.name = series_id
    returns_dict[series_id] = log_ret
    print(f"  {series_id}: {len(log_ret)} observations, "
          f"{log_ret.index[0].date()} to {log_ret.index[-1].date()}")

print(f"\nTotal series loaded: {len(returns_dict)}")

## 2. Load Chronos-2 Bolt Model

In [ ]:
from chronos import ChronosBoltPipeline

print("Loading Chronos-2 Bolt model...")
try:
    pipeline = ChronosBoltPipeline.from_pretrained(
        "amazon/chronos-bolt-base",
        device_map=device,
        torch_dtype=torch.float32,
    )
    print(f"Chronos-2 Bolt (base) loaded on {device}")
except Exception as e:
    print(f"Base model failed: {e}")
    print("Trying small model as fallback...")
    pipeline = ChronosBoltPipeline.from_pretrained(
        "amazon/chronos-bolt-small",
        device_map=device,
        torch_dtype=torch.float32,
    )
    print(f"Chronos-2 Bolt (small) loaded on {device}")

## 3. Run VaR/ES Forecasting

In [ ]:
def fit_t_to_quantiles(q_vals, q_levels):
    """Fit Student-t to quantile points via quantile matching."""
    def objective(params):
        loc, scale, df = params
        if scale <= 0 or df <= 2:
            return 1e10
        predicted = t_dist.ppf(q_levels, df=df, loc=loc, scale=scale)
        return np.sum((predicted - q_vals) ** 2)

    loc0 = np.median(q_vals)
    scale0 = max((q_vals[-1] - q_vals[0]) / 4, 1e-8)
    result = minimize(objective, [loc0, scale0, 5.0],
                      method='Nelder-Mead', options={'maxiter': 2000})
    return result.x


def extract_var_es_from_quantiles(q_vals):
    """Fit Student-t to Bolt quantile outputs, extract VaR/ES analytically."""
    params = fit_t_to_quantiles(q_vals, Q_LEVELS)
    loc_p, scale_p, df_p = params
    df_p = max(df_p, 2.01)
    scale_p = max(scale_p, 1e-10)

    row = {}
    for alpha in VAR_ALPHAS:
        alpha_str = {0.01: "01", 0.025: "025", 0.05: "05"}[alpha]
        row[f"var_{alpha_str}"] = float(t_dist.ppf(alpha, df=df_p, loc=loc_p, scale=scale_p))

    var_es = row["var_025"]
    t_val = (var_es - loc_p) / scale_p
    pdf_val = t_dist.pdf(t_val, df=df_p)
    row["es_025"] = float(loc_p + scale_p * (
        -pdf_val / ES_ALPHA * (df_p + t_val**2) / (df_p - 1)
    ))

    row["fm_median"] = float(q_vals[4])
    row["fm_std"] = float((q_vals[-1] - q_vals[0]) / 2.563)
    return row


def run_chronos(pipeline, returns, oos_dates, series_id, save_path):
    """Run Chronos-2 Bolt VaR/ES forecasts for one series."""
    results = []
    n_total = len(oos_dates)

    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < FM_CONTEXT:
            continue

        context = returns.iloc[loc - FM_CONTEXT:loc].values.astype(np.float64)
        y_true = float(returns.iloc[loc])

        try:
            ctx_tensor = torch.tensor(context, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                quantiles = pipeline.predict(ctx_tensor, prediction_length=FM_HORIZON)
            q_vals = quantiles[0, :, 0].cpu().numpy()
        except Exception as e:
            if i < 5:
                print(f"    Forecast error at {date}: {e}")
            continue

        row = {"date": str(date.date()), "y_true": y_true}
        row.update(extract_var_es_from_quantiles(q_vals))
        results.append(row)

        if len(results) % 100 == 0:
            pd.DataFrame(results).to_csv(save_path, index=False, float_format="%.8f")

        if (i + 1) % 100 == 0 or i == n_total - 1:
            print(f"    [{i+1}/{n_total}] {series_id}")

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False, float_format="%.8f")
    return df

In [ ]:
# ── Main forecasting loop ─────────────────────────────────
print("=" * 60)
print("CHRONOS-2 BOLT VaR/ES FORECASTING")
print(f"Context: {FM_CONTEXT} | Quantiles: 9 (10th-90th) | Stride: {STRIDE}")
print("=" * 60)

all_results = {}

for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        print(f"\nSKIP: {series_id} (not loaded)")
        continue

    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]

    out_path = OUT_DIR / f"Chronos-2_{series_id}.csv"

    if out_path.exists():
        existing = pd.read_csv(out_path)
        if len(existing) >= len(oos_dates):
            print(f"\n[{series_id}] Already done ({len(existing)} rows), skipping.")
            all_results[series_id] = existing
            continue

    print(f"\n[{series_id}] {len(returns)} total obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_chronos(pipeline, returns, oos_dates, series_id, out_path)
    elapsed = time.time() - t0

    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {elapsed:.1f}s")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

## 4. Visualisation — VaR Breach Plot

In [ ]:
# Plot VaR breach examples for all index series
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue

    df_plot = all_results[series_id].copy()
    if "date" in df_plot.columns:
        df_plot["date"] = pd.to_datetime(df_plot["date"])
    var_col = "var_01"
    if var_col not in df_plot.columns:
        continue

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4,
            color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6,
            color="coral", linestyle="--", label="Chronos-2 VaR 1%")

    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"],
                   df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")

    n_breach = breaches.sum()
    pct_breach = 100 * n_breach / len(df_plot)
    ax.set_title(f"Chronos-2 VaR 1% — {series_id}  "
                 f"(breaches: {n_breach}/{len(df_plot)} = {pct_breach:.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25),
              ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

## 5. Download Results

In [ ]:
# Download all result CSVs
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("Chronos-2_*.csv")):
        zf.write(f, f.name)

zip_buffer.seek(0)
with open("Chronos2_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())

files.download("Chronos2_results.zip")
print("Download started: Chronos2_results.zip")